# 01.02 — LLM vs SLM: Definisi & Garis Pemisah

**Tujuan**: punya definisi konkret (bukan vague) untuk LLM dan SLM, paham di mana garis pemisahnya, dan kapan satu lebih cocok dari yang lain.

**Prasyarat**: notebook 01.01 lulus (paham apa itu language model).

**Catatan**: notebook ini sebagian besar narasi + tabel. Sedikit kode untuk visualisasi.

---

## TL;DR

- **LLM** = language model dengan params **≥ ~7 milyar** (consensus longgar). Biasanya butuh GPU besar atau API hosted untuk inference.
- **SLM** = language model dengan params **< ~7 milyar**. Bisa muat di mesin biasa (laptop, edge device) — sering dengan quantization.

Tidak ada threshold formal. Yang penting bukan angka tepat, tapi **konsekuensi praktisnya**: bisa jalan di mana, biaya berapa, kemampuan apa.

## 0. Bootstrap (jalankan pertama)

Cell standar yang bikin notebook jalan di lokal & Colab. Notebook ini import `from utils.plotting`, jadi cell ini WAJIB supaya `sys.path` benar.

In [ ]:
import sys
from pathlib import Path

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    REPO_NAME = "llm-vs-slm-lab"
    REPO_URL = "https://github.com/rizkyhaksono/llm-vs-slm-lab.git"  # <-- ganti setelah push
    if not Path(REPO_NAME).exists():
        !git clone {REPO_URL}
    %cd {REPO_NAME}
    !pip install -q torch --index-url https://download.pytorch.org/whl/cpu
    !pip install -q -r requirements.txt

repo_root = None
for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "requirements.txt").exists():
        repo_root = candidate
        break
assert repo_root is not None, "Tidak ketemu root repo (requirements.txt). Jalankan dari dalam repo."
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
print(f"IN_COLAB={IN_COLAB}, repo_root={repo_root}")

## 1. Garis pemisah: definisi kasar vs definisi praktis

**Definisi kasar (dari paper / blog)**:

| Kategori | Range params (umum) |
|---|---|
| Tiny LM | < 1 B |
| Small LM (SLM) | 1 B – 7 B |
| Medium LM | 7 B – 70 B |
| Large LM (LLM) | > 70 B |

Tapi ini tidak terlalu berguna karena "small" terus geser tiap tahun.

**Definisi praktis (dipakai di repo ini)**:

| Kategori | Definisi | Contoh |
|---|---|---|
| **SLM** | Bisa di-inference di **laptop CPU** (mungkin dengan quantization) dalam waktu masuk akal | DistilBERT (66M), SmolLM2 (135M), TinyLlama (1.1B Q4), Phi-3-mini (3.8B Q4) |
| **LLM** | Butuh **GPU mahal atau API hosted** untuk inference, atau setidaknya 32GB+ RAM tinggi | Llama-3.1-8B/70B, GPT-4, Claude Opus, Gemini Pro |

Definisi praktis lebih useful karena keputusan engineering kamu didorong oleh **"bisa jalan di mana"** dan **"siapa yang bayar inference"** — bukan oleh angka params di paper.

## 2. Visualisasi spektrum params

Ini chart sederhana untuk kalibrasi mental: di mana model-model populer berada di garis params.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

from utils.plotting import COLORS, setup_style

setup_style()

# (nama, params dalam juta, color_key)
models = [
    ("DistilBERT", 66, "slm_fp32"),
    ("SmolLM2-135M", 135, "slm_fp32"),
    ("SmolLM2-360M", 360, "slm_fp32"),
    ("TinyLlama", 1_100, "slm_q4"),
    ("SmolLM2-1.7B", 1_700, "slm_fp32"),
    ("Phi-3-mini", 3_800, "slm_q4"),
    ("Llama-3.1-8B", 8_000, "llm"),
    ("Llama-3.3-70B", 70_000, "llm"),
    ("GPT-4 (rumor)", 1_700_000, "llm"),
]

fig, ax = plt.subplots(figsize=(10, 4))
for name, params_m, color_key in models:
    ax.scatter(params_m, 0, s=200, color=COLORS[color_key], edgecolor="black", zorder=3)
    ax.annotate(
        name,
        (params_m, 0),
        xytext=(0, 15),
        textcoords="offset points",
        ha="center",
        fontsize=9,
        rotation=30,
    )

ax.set_xscale("log")
ax.set_xlabel("Parameters (juta, log scale)")
ax.set_yticks([])
ax.set_title("Spektrum Language Model: dari kecil ke besar")
ax.axvline(7_000, linestyle="--", color="gray", alpha=0.6)
ax.text(7_000, 0.4, "~7B (SLM/LLM frontier)", ha="center", fontsize=9, color="gray")
ax.set_ylim(-1, 1.5)
ax.grid(True, axis="x", alpha=0.3)

ax.legend(
    handles=[
        Patch(color=COLORS["slm_fp32"], label="SLM (fp32, transformers)"),
        Patch(color=COLORS["slm_q4"], label="SLM (Q4 quantized, llama.cpp)"),
        Patch(color=COLORS["llm"], label="LLM (API)"),
    ],
    loc="upper left",
)
plt.tight_layout()
plt.show()

## 3. Perbandingan formal (tabel)

Tiga axis utama yang membedakan:

### 3.1. Kapasitas / Kemampuan

| Aspek | SLM | LLM |
|---|---|---|
| Multi-step reasoning | Lemah | Kuat |
| Coding (non-trivial) | Lemah | Kuat |
| World knowledge | Terbatas | Luas |
| Multilingual | Sering monolingual | Multi-bahasa native |
| Klasifikasi spesifik (setelah fine-tune) | **Bisa mengalahkan LLM** | Overkill |
| Extraction / structured output | OK setelah fine-tune | OK out-of-the-box |

Insight penting: **SLM yang di-fine-tune di domain spesifik bisa mengalahkan LLM general** untuk task itu. Contoh: DistilBERT yang di-fine-tune sentiment Bahasa biasanya akurasinya menyamai atau lebih dari Llama-3.1-8B few-shot — di task itu, di Bahasa itu.

### 3.2. Operational

| Aspek | SLM (lokal) | LLM (API) |
|---|---|---|
| Setup awal | Install lib, download model (5–700 MB) | API key, kirim HTTP request |
| Hardware | Laptop CPU OK (dengan model kecil/quantized) | GPU mahal kalau self-host, atau API |
| Latency | 1–10 detik di CPU laptop | 200ms (Groq) – 3s (lainnya) |
| Biaya per request | $0 (setelah listrik) | $0.0001 – $0.05 |
| Biaya operasional | Engineer maintenance, infra | API bill |
| Privacy | Data tetap lokal | Data ke server provider |
| Offline-able | Ya | Tidak |
| Update model | Manual | Otomatis (kadang silent) |

### 3.3. Skala

| Aspek | SLM | LLM |
|---|---|---|
| Training cost | $1k – $1M | $1M – $100M+ |
| Training data scale | 100B – 1T tokens | 1T – 15T tokens |
| Inference cost (per million tokens) | <$0.01 (lokal CPU) | $0.05 – $50 |
| Bisa di-fine-tune di laptop? | Ya (DistilBERT, SmolLM2 kecil) | Tidak (butuh GPU cluster) |

## 4. Mitos yang perlu dilurusin

**Mitos 1: "Lebih besar selalu lebih baik"**

Tidak. Untuk task spesifik dengan training data spesifik, SLM fine-tuned sering lebih akurat **dan** lebih cepat **dan** lebih murah. "Lebih besar lebih baik" benar di task **general / open-ended** tanpa fine-tune.

**Mitos 2: "SLM = LLM yang dikecilin"**

Tidak persis. Banyak SLM modern **dilatih dari nol** (Phi-3, SmolLM2) dengan filosofi: data quality > data quantity. Microsoft Phi-3 paper argue bahwa data textbook-quality bikin model 3.8B kompetitif dengan model 30B di task-task tertentu.

**Mitos 3: "LLM bisa segalanya, SLM cuma toy"**

Untuk klasifikasi sentiment di komentar Bahasa, DistilBERT yang ditraining 30 menit di laptop bisa lebih akurat dari Llama-3.1-8B zero-shot. SLM bukan toy — SLM **specialist**. LLM **generalist**.

**Mitos 4: "Quantization gratis"**

Q4 (4-bit) memang hemat ~8x storage vs fp32, dan ~3x lebih cepat di CPU. Tapi quality bisa drop di task delicate (math, code). Selalu A/B test.

## 5. Kapan pakai apa — preview dari modul 05

Decision framework lengkap di modul 05. Tapi rule of thumb yang bisa kamu pakai sekarang:

In [ ]:
# Decision rule sederhana — bukan production code, cuma mental model.
def quick_recommend(
    is_data_sensitive: bool,
    needs_complex_reasoning: bool,
    is_simple_classification_or_extraction: bool,
    high_volume_per_day: bool,
    needs_low_latency_under_500ms: bool,
    needs_offline: bool,
) -> str:
    if needs_offline:
        return "SLM lokal (no choice)"
    if is_data_sensitive:
        return "SLM lokal / fine-tuned (privacy hard requirement)"
    if needs_complex_reasoning:
        return "LLM API (SLM kecil tidak cukup)"
    if is_simple_classification_or_extraction:
        return "SLM fine-tuned (lebih cepat, akurat, murah)"
    if high_volume_per_day and needs_low_latency_under_500ms:
        return "Groq atau SLM lokal"
    return "LLM API default — start simple"

# Skenario contoh: sentiment analysis 10k komentar/hari
print(quick_recommend(
    is_data_sensitive=False,
    needs_complex_reasoning=False,
    is_simple_classification_or_extraction=True,
    high_volume_per_day=True,
    needs_low_latency_under_500ms=True,
    needs_offline=False,
))

## Refleksi & insight

Setelah notebook ini kamu harusnya bisa:

1. Beda LLM dan SLM dalam 1 kalimat ("bisa jalan di mana, siapa yang bayar inference, kemampuan apa").
2. Sebutkan 3 task di mana SLM fine-tuned biasanya menang dari LLM general.
3. Sebutkan 3 task di mana SLM tidak akan cukup, harus LLM.
4. Tahu kenapa "lebih besar lebih baik" tidak selalu benar.

## Latihan mandiri (opsional)

1. **Cari 1 paper SLM 2024–2026** (mis. SmolLM2, Phi-3, Gemma-2). Catat: ukuran params, training data scale, klaim utama paper. Bandingkan dengan klaim LLM seperti Llama-3.
2. **Pikirkan task NLP yang kamu hadapi di kerjaan / project pribadi**. Pakai `quick_recommend()` di atas — apa keluarnya? Setuju atau tidak? Kalau tidak, kenapa?

## Lanjut

Cukup teori. Sekarang kita rasakan langsung: [02-inference-perbandingan/01_hello_groq_llm.ipynb](../02-inference-perbandingan/01_hello_groq_llm.ipynb).